In [ ]:
import pandas as pd
import numpy as np


In [ ]:
def approximate_split_finder(X_feature, g, h, num_bins=10):
    """
    Finds the best split point for a single feature using an approximate algorithm.
    
    Parameters:
    - X_feature: 1D array of feature values for the current node.
    - g: 1D array of first-order gradients for the samples.
    - h: 1D array of second-order hessians for the samples.
    - num_bins: Number of quantiles/bins to propose candidate splits.
    
    Returns:
    - best_split: The optimal feature threshold value to split on.
    - max_gain: The maximum gain achieved by the split.
    """
    # 1. Sort feature values and align gradients and hessians
    sort_idx = np.argsort(X_feature)
    X_sorted = X_feature[sort_idx]
    g_sorted = g[sort_idx]
    h_sorted = h[sort_idx]
    
    # 2. Propose candidate splits based on quantiles (percentiles)
    # Equivalent to finding approximate percentiles across the feature values
    percentiles = np.linspace(0, 100, num_bins + 1)[1:-1]
    candidate_splits = np.percentile(X_sorted, percentiles)
    candidate_splits = np.unique(candidate_splits)  # Remove duplicate split values
    
    # Pre-calculate total gradient and hessian for the node
    G_total = np.sum(g_sorted)
    H_total = np.sum(h_sorted)
    
    best_gain = -float('inf')
    best_split = None
    
    # Define regularization parameter (lambda) for gain calculation
    reg_lambda = 1.0 
    
    # Helper to calculate tree node score (reduction in loss)
    def calc_score(G, H):
        return (G ** 2) / (H + reg_lambda)
    
    # Base score of the unsplit node
    root_score = calc_score(G_total, H_total)
    
    # 3. Evaluate candidate split points
    for split_val in candidate_splits:
        # Split samples based on candidate threshold
        left_mask = X_sorted <= split_val
        
        G_L = np.sum(g_sorted[left_mask])
        H_L = np.sum(h_sorted[left_mask])
        
        G_R = G_total - G_L
        H_R = H_total - H_L
        
        # Calculate Gain: 0.5 * [Score(Left) + Score(Right) - Score(Parent)]
        gain = 0.5 * (calc_score(G_L, H_L) + calc_score(G_R, H_R) - root_score)
        
        if gain > best_gain:
            best_gain = gain
            best_split = split_val

    return best_split, max(best_gain, 0.0)
